# Shadow Supervisor — Training Run Notebook

This notebook reproduces the training and evaluation pipeline for Shadow Supervisor.

It runs scenario generation, expert dataset generation, environment-connected RL training, hardened evaluation, and plot generation.


## 0. Setup

If running in Google Colab, set `RUN_IN_COLAB = True` and update `REPO_URL`.

If running locally from the project root, keep `RUN_IN_COLAB = False`.


In [ ]:
RUN_IN_COLAB = False
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/shadow-supervisor-openenv.git"
PROJECT_DIR = "shadow-supervisor-openenv"

print("RUN_IN_COLAB:", RUN_IN_COLAB)
print("REPO_URL:", REPO_URL)


In [ ]:
import os
from pathlib import Path

if RUN_IN_COLAB:
    if not Path(PROJECT_DIR).exists():
        !git clone $REPO_URL
    os.chdir(PROJECT_DIR)

print("Current directory:", os.getcwd())
!ls


## 1. Install dependencies


In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt


## 2. Verify OpenEnv compliance


In [ ]:
!python -m pip show openenv-core
!python scripts/verify_openenv_compliance.py


## 3. Generate scenario data


In [ ]:
!python training/build_real_data.py
!ls -lh data | head


## 4. Build expert dataset


In [ ]:
!python training/build_expert_dataset.py


## 5. Run TRL/SFT-compatible training script

This is the fallback training path. The main training proof is the environment-connected RL run in the next step.


In [ ]:
!python training/train_trl.py


## 6. Run environment-connected RL training


In [ ]:
!python training/train_env_rl.py
!ls -lh outputs | grep -E 'rl_policy|training|reward|safety' || true


## 7. Run hardened evaluation


In [ ]:
!python evaluation/run_eval_hardened.py
!cat outputs/policy_comparison_hardened.csv


## 8. Generate final plots


In [ ]:
!python evaluation/generate_winning_plots.py
!ls -lh outputs/*.png


## 9. Display final policy comparison


In [ ]:
import pandas as pd

df = pd.read_csv("outputs/policy_comparison_hardened.csv")
df


## 10. Display training plots


In [ ]:
from IPython.display import Image, display

plot_files = [
    "outputs/baseline_vs_trained.png",
    "outputs/winning_rl_reward_curve.png",
    "outputs/winning_training_loss.png",
    "outputs/winning_unsafe_approval_rate.png",
    "outputs/rl_safety_curve.png",
]

for plot in plot_files:
    print("\n" + plot)
    display(Image(filename=plot))


## Final interpretation

The expected final result is:

- Naive supervisor fails.
- Training candidate partially improves but remains unsafe.
- Spam-all-actions baseline fails, proving reward is hard to game.
- RL-trained supervisor reaches high reward, 100% success, and 0% unsafe approval.
